# Tiempos procesales por tipo de delito sentenciado

Cuatro tablas independientes con estructura identica para poder
compararlas: el total de la base como referencia, delitos contra la
propiedad, homicidios y delitos sexuales. Cada tabla trae los mismos nueve
tiempos procesales con minimo, maximo, promedio, mediana y N.

El total no es la suma de las otras tres. Incluye procesos sin condena,
absueltos y condenados por delitos fuera de los tres grupos, y un proceso
condenado por hurto y homicidio esta contado en dos de las tablas por
delito.

La clasificacion es por delito sentenciado (`DelitoSentencia`), con los
articulos del listado de referencia. Propiedad es la categoria completa de
crimen a la propiedad; homicidios y delitos sexuales son dos subgrupos de
crimen violento que el listado no separa, asi que la particion queda
explicita en la celda de abajo y documentada en el Excel.

Limpieza de las duraciones: se excluyen las diferencias negativas, porque
una fecha final anterior a la inicial es un dato inconsistente y no una
duracion, y despues el 1% mas alto de cada variable. Se excluyen y no se
truncan, porque reemplazar por el valor del percentil seria inventar el
dato. El umbral se calcula una sola vez por variable sobre la base total y
se aplica igual a las cuatro tablas.

Dos consecuencias de lectura, anotadas tambien dentro del Excel: la
columna Maximo pasa a ser el percentil 99 y no el caso mas lento del
sistema, y el Minimo va a ser 0 en casi todas las filas.

Solo los procesos con condena tienen delito sentenciado, asi que las tres
tablas por delito se calculan sobre ese universo y no sobre todas las
capturas.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

DESCARGAS = Path.home() / "Downloads"

RUTA_BASE = DESCARGAS / "base_final.parquet"
RUTA_BASE_MUESTRA = DESCARGAS / "muestra_revision.csv"
RUTA_EXCEL = DESCARGAS / "tiempos_por_tipo_delito.xlsx"

COL = {
    "captura": "FECHA",
    "legalizacion": "FechaLegalizacion",
    "imputacion": "FechaImputacion",
    "medida": "FechaMedida",
    "acusacion": "FechaAcusacion",
    "sentencia": "FechaSentencia",
    "boleta_salida": "FechaBoletaSalida",
}

COL_PROCESO = "radicado"
COL_ID = "IDENTIFICACION"
COL_DELITO = "DelitoSentencia"

DESCARTAR_NEGATIVOS = True

PERCENTIL_CORTE = 0.99

GRUPO_TOTAL = "Total de la base"

GRUPOS = {
    "Delitos Contra la Propiedad": {"239", "240", "244", "246", "249", "250"},
    "Crimen Violento": {"103", "104A", "104B", "105", "106", "108", "111", "119" , "128" , "135", "138", "139", "139A", "141", "141A", "141B", "205", "206", "208",
            "209", "210A", "213", "213A", "214", "217", "218", "219A", "229" , "230", "230A" , "229A"},
    "Drogas y Armas": { "376" , "365" , "366"},
}

COLORES = {
    GRUPO_TOTAL: "EDEDED",
    "Delitos contra la propiedad": "DDEBF7",
    "Delitos violentos": "FCE4D6",
    "Drogas y Armas": "E2EFDA",
}

In [ ]:
if RUTA_BASE.exists():
    data = pd.read_parquet(RUTA_BASE)
    origen = RUTA_BASE.name
else:
    data = pd.read_csv(RUTA_BASE_MUESTRA, encoding="utf-8-sig")
    origen = RUTA_BASE_MUESTRA.name


print(f"Base cargada: {origen}  ({len(data):,} filas en formato largo)")

faltantes = [c for c in list(COL.values()) + [COL_PROCESO, COL_DELITO]
             if c not in data.columns]
if faltantes:
    raise KeyError(f"Faltan columnas en la base: {faltantes}")


for columna in COL.values():
    data[columna] = pd.to_datetime(data[columna], errors="coerce")


data["articulo_sentencia"] = data[COL_DELITO].str.extract(
    r"Art[ií]culo\s+(\d+[A-Z]?)\.", flags=re.IGNORECASE
)[0]

data["grupo"] = np.select(
    [data["articulo_sentencia"].isin(arts) for arts in GRUPOS.values()],
    list(GRUPOS),
    default=None,
)

print("\nFilas por grupo (formato largo):")
print(data["grupo"].value_counts(dropna=False).to_string())

fuera = (
    data.loc[data["grupo"].isna() & data["articulo_sentencia"].notna(),
             ["articulo_sentencia", COL_DELITO]]
    .value_counts()
    .head(15)
)
if len(fuera):
    print("\nDelitos sentenciados fuera de los tres grupos (top 15):")
    print(fuera.to_string())

## 2. Duraciones, calculadas una sola vez por proceso

Los intervalos son atributos del proceso y no del delito. Se calculan
sobre la base deduplicada por radicado y despues se le pega el grupo, para
no computar dos veces el mismo intervalo cuando un proceso tiene varios
delitos.

In [ ]:
INTERVALOS = [
    ("Captura → Legalización",               "dias_captura_legalizacion",       "captura",      "legalizacion"),
    ("Captura → Imputación",                 "dias_captura_imputacion",         "captura",      "imputacion"),
    ("Legalización → Imputación",            "dias_legalizacion_imputacion",    "legalizacion", "imputacion"),
    ("Imputación → Medida de aseguramiento", "dias_imputacion_medida",          "imputacion",   "medida"),
    ("Medida de aseguramiento → Acusación",  "dias_medida_acusacion",           "medida",       "acusacion"),
    ("Acusación → Sentencia",                "dias_acusacion_sentencia",        "acusacion",    "sentencia"),
    ("Captura → Sentencia",                  "dias_captura_sentencia",          "captura",      "sentencia"),
    ("Imputación → Sentencia",               "dias_imputacion_sentencia",       "imputacion",   "sentencia"),
    ("Captura → Boleta de salida",           "dias_captura_boleta_salida",      "captura",      "boleta_salida"),
    ("Legalización → Boleta de salida",      "dias_legalizacion_boleta_salida", "legalizacion", "boleta_salida"),
]

proc = data.drop_duplicates(subset=[COL_PROCESO]).copy()

VARIABLES = [nombre for _, nombre, _, _ in INTERVALOS]
VARIABLES_BRUTAS = [f"{v}_bruto" for v in VARIABLES]

for _, nombre, ini, fin in INTERVALOS:
    proc[f"{nombre}_bruto"] = (proc[COL[fin]] - proc[COL[ini]]).dt.days


umbrales, exclusiones = {}, {}

for _, nombre, _, _ in INTERVALOS:
    bruto = proc[f"{nombre}_bruto"]

    negativos = bruto < 0
    valores = bruto.where(~negativos) if DESCARTAR_NEGATIVOS else bruto

    if PERCENTIL_CORTE is not None and valores.notna().any():
        umbral = valores.quantile(PERCENTIL_CORTE)
    else:
        umbral = np.nan

    sobre_umbral = valores.notna() & (valores > umbral) if pd.notna(umbral) else pd.Series(False, index=proc.index)

    proc[nombre] = valores.where(~sobre_umbral)

    umbrales[nombre] = umbral
    exclusiones[nombre] = {
        "válidas antes de limpiar": int(bruto.notna().sum()),
        "negativas": int(negativos.sum()),
        f"sobre el p{int(PERCENTIL_CORTE * 100) if PERCENTIL_CORTE else 0}": int(sobre_umbral.sum()),
        "umbral (días)": round(umbral, 1) if pd.notna(umbral) else np.nan,
        "válidas finales": int(proc[nombre].notna().sum()),
    }


print("Limpieza de cada variable (umbral global, calculado sobre la base total):\n")
print(pd.DataFrame(exclusiones).T.to_string())

COLS_ARRASTRE = [COL_PROCESO, COL_ID] + [COL[c] for c in COL] + VARIABLES + VARIABLES_BRUTAS

pares = data.loc[data["grupo"].notna(), [COL_PROCESO, "grupo"]].drop_duplicates()

por_grupo = pares.merge(proc[COLS_ARRASTRE], on=COL_PROCESO, how="left")

total = proc[COLS_ARRASTRE].assign(grupo=GRUPO_TOTAL)

procesos_grupo = pd.concat([total, por_grupo], ignore_index=True)

TABLAS = [GRUPO_TOTAL] + list(GRUPOS)

print(f"\nProcesos únicos                : {len(proc):,}")
print(f"Procesos con delito clasificado: {pares[COL_PROCESO].nunique():,}")
print(f"En más de un grupo             : "
      f"{(pares.groupby(COL_PROCESO).size() > 1).sum():,}")


print("\nProcesos por tabla:")
print(procesos_grupo["grupo"].value_counts().reindex(TABLAS).to_string())

In [ ]:
def tabla_grupo(grupo: str) -> pd.DataFrame:
    sub = procesos_grupo[procesos_grupo["grupo"] == grupo]

    filas = []
    for etiqueta, nombre, _, _ in INTERVALOS:
        s = sub[nombre].dropna()
        filas.append({
            "Tiempo procesal": etiqueta,
            "Mínimo": s.min() if len(s) else np.nan,
            "Máximo": s.max() if len(s) else np.nan,
            "Promedio": round(s.mean(), 1) if len(s) else np.nan,
            "Mediana": s.median() if len(s) else np.nan,
            "N": len(s),
        })

    return pd.DataFrame(filas).set_index("Tiempo procesal")


tablas = {}

for i, grupo in enumerate(TABLAS, start=1):
    tablas[grupo] = tabla_grupo(grupo)
    print(f"\nTABLA {i} — {grupo.upper()}  "
          f"({procesos_grupo['grupo'].eq(grupo).sum():,} procesos)\n")
    print(tablas[grupo].to_string())

## 4. Excel

Las tablas van una debajo de otra en la misma hoja, con la misma
estructura y separadas por espacio en blanco. Los estadisticos son
formulas que apuntan a la hoja de datos de su grupo, no valores pegados.

In [ ]:
try:
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    from openpyxl.utils import get_column_letter
except ImportError as e:
    raise ImportError(
        "Falta openpyxl. En PowerShell:\n"
        r"  uv pip install --python C:\Users\DiDi\.venv\Scripts\python.exe openpyxl"
        "\ny después reiniciá el kernel."
    ) from e


FUENTE = "Arial"
NORMAL = Font(name=FUENTE, size=10)
NEGRITA = Font(name=FUENTE, size=10, bold=True)
TITULO = Font(name=FUENTE, size=12, bold=True)
SUBTITULO = Font(name=FUENTE, size=9, italic=True, color="595959")

BORDE_TENUE = Border(bottom=Side(style="thin", color="BFBFBF"))
BORDE_CABECERA = Border(bottom=Side(style="medium", color="808080"))

CABECERAS = ["Tiempo procesal", "Mínimo", "Máximo", "Promedio", "Mediana", "N"]

HOJA_DATOS = {
    GRUPO_TOTAL: "datos_total",
    "Delitos contra la propiedad": "datos_propiedad",
    "Delitos violentos": "datos_violentos",
    "Otros": "datos_otros",
}

wb = Workbook()

COLS_DATOS = (
    [COL_PROCESO, COL_ID, "grupo"] + [COL[c] for c in COL]
    + VARIABLES + VARIABLES_BRUTAS
)
letra = {c: get_column_letter(i + 1) for i, c in enumerate(COLS_DATOS)}
ultima_fila = {}

for grupo, nombre_hoja in HOJA_DATOS.items():
    hoja = wb.create_sheet(nombre_hoja)
    hoja.append(COLS_DATOS)

    sub = procesos_grupo.loc[procesos_grupo["grupo"] == grupo, COLS_DATOS]

    for fila in sub.itertuples(index=False):
        hoja.append([
            v.to_pydatetime() if isinstance(v, pd.Timestamp)
            else (None if pd.isna(v) else v)
            for v in fila
        ])

    for celda in hoja[1]:
        celda.font = NEGRITA
        celda.fill = PatternFill("solid", fgColor=COLORES[grupo])
        celda.alignment = Alignment(horizontal="center", wrap_text=True)
        celda.border = BORDE_CABECERA

    for fila in hoja.iter_rows(min_row=2):
        for celda in fila:
            celda.font = NORMAL

    for c in COLS_DATOS:
        hoja.column_dimensions[letra[c]].width = 18
        if c in COL.values():
            for f in range(2, len(sub) + 2):
                hoja[f"{letra[c]}{f}"].number_format = "yyyy-mm-dd"

    hoja.auto_filter.ref = f"A1:{get_column_letter(len(COLS_DATOS))}{max(len(sub) + 1, 1)}"
    hoja.freeze_panes = "D2"

    ultima_fila[grupo] = max(len(sub) + 1, 2)

In [ ]:
ws = wb["Sheet"]
ws.title = "Tablas"

ws["A1"] = "Tiempos procesales por tipo de delito sentenciado"
ws["A1"].font = Font(name=FUENTE, size=14, bold=True)
pct = int(PERCENTIL_CORTE * 100) if PERCENTIL_CORTE else None

ws["A2"] = (
    "Días entre etapas. La primera tabla es la base completa y sirve de "
    "referencia; las otras tres son subconjuntos por delito sentenciado. "
    + (f"Se excluyen las duraciones negativas y el 1% más alto de cada "
       f"variable, con umbral único calculado sobre la base total: por eso "
       f"la columna Máximo equivale al percentil {pct}. " if pct else "")
    + "Fuente: " + origen
)
ws["A2"].font = SUBTITULO

ws.column_dimensions["A"].width = 40
for c in "BCDEF":
    ws.column_dimensions[c].width = 14


fila = 4

for i, grupo in enumerate(TABLAS, start=1):
    n_procesos = int(procesos_grupo["grupo"].eq(grupo).sum())
    hoja_datos = HOJA_DATOS[grupo]
    fin = ultima_fila[grupo]

    ws[f"A{fila}"] = f"Tabla {i} — {grupo}"
    ws[f"A{fila}"].font = TITULO
    fila += 1

    ws[f"A{fila}"] = (
        f"{n_procesos:,} procesos, sin filtrar por delito"
        if grupo == GRUPO_TOTAL
        else f"{n_procesos:,} procesos con condena por este tipo de delito"
    )
    ws[f"A{fila}"].font = SUBTITULO
    fila += 1

    for j, cabecera in enumerate(CABECERAS):
        celda = ws.cell(row=fila, column=j + 1, value=cabecera)
        celda.font = NEGRITA
        celda.fill = PatternFill("solid", fgColor=COLORES[grupo])
        celda.alignment = Alignment(horizontal="center", wrap_text=True)
        celda.border = BORDE_CABECERA
    fila += 1

    for etiqueta, variable, _, _ in INTERVALOS:
        L = letra[variable]
        rango = f"{hoja_datos}!${L}$2:${L}${fin}"

        ws[f"A{fila}"] = etiqueta
        ws[f"B{fila}"] = f'=IFERROR(MIN({rango}),"")'
        ws[f"C{fila}"] = f'=IFERROR(MAX({rango}),"")'
        ws[f"D{fila}"] = f'=IFERROR(AVERAGE({rango}),"")'
        ws[f"E{fila}"] = f'=IFERROR(MEDIAN({rango}),"")'
        ws[f"F{fila}"] = f"=COUNT({rango})"

        for c in "ABCDEF":
            ws[f"{c}{fila}"].font = NORMAL
            ws[f"{c}{fila}"].border = BORDE_TENUE
        for c in "BCDE":
            ws[f"{c}{fila}"].number_format = "#,##0.0"
        ws[f"F{fila}"].number_format = "#,##0"

        fila += 1

    fila += 2


ws.freeze_panes = "A4"
print(f"Tablas escritas hasta la fila {fila}.")

In [ ]:
doc = wb.create_sheet("Documentacion")
doc["A1"] = "Documentación del cálculo"
doc["A1"].font = TITULO

lineas = [
    ("Archivo de origen", origen),
    ("Filas en formato largo", f"{len(data):,}"),
    ("Procesos únicos", f"{len(proc):,}"),
    ("Procesos con delito sentenciado clasificado", f"{pares[COL_PROCESO].nunique():,}"),
    ("", ""),
    ("CRITERIOS", ""),
    ("Tabla total",
     f"'{GRUPO_TOTAL}' incluye TODOS los procesos, sin filtrar por delito: "
     "los sin condena, los absueltos y los condenados por delitos fuera de "
     "los tres grupos. No es la suma de las otras tres tablas."),
    ("Variable de clasificación",
     f"{COL_DELITO} (delito de la sentencia), por número de artículo. Solo los "
     "procesos con condena tienen este dato, así que las tres tablas por "
     "delito se calculan sobre ese universo."),
    ("Unidad de análisis",
     "El proceso (radicado). Las duraciones se calculan una sola vez por "
     "proceso, aunque la base venga en formato largo con una fila por delito."),
    ("Procesos en más de un grupo",
     f"{(pares.groupby(COL_PROCESO).size() > 1).sum():,} procesos fueron "
     "condenados por delitos de más de un grupo y aparecen en más de una "
     "tabla. Las tablas no son excluyentes entre sí."),
    ("Cálculo", "días = fecha final − fecha inicial."),
    ("Fechas faltantes",
     "No se imputa nada. Si falta una de las dos fechas, la observación se "
     "excluye únicamente de ese intervalo; la columna N refleja las "
     "observaciones válidas de cada fila."),
    ("Diferencias negativas",
     "Excluidas: una fecha final anterior a la inicial es un dato "
     "inconsistente, no una duración." if DESCARTAR_NEGATIVOS
     else "Se conservan tal cual."),
    ("Recorte del 1% más alto",
     (f"Se excluye el {int((1 - PERCENTIL_CORTE) * 100)}% superior de cada "
      f"variable. No se trunca al valor del percentil: truncar sería inventar "
      f"un dato, la misma objeción que con los negativos. El umbral se calcula "
      f"UNA vez por variable sobre la base total y se aplica igual a las "
      f"cuatro tablas, para que se midan con la misma vara; el costo es que un "
      f"grupo genuinamente más lento pierde más observaciones que el resto. "
      f"Consecuencia de lectura: la columna Máximo es el percentil "
      f"{int(PERCENTIL_CORTE * 100)}, no el caso más lento del sistema."
      ) if PERCENTIL_CORTE else "No se aplica."),
    ("Orden de la limpieza",
     "Primero se sacan los negativos y después se calcula el percentil sobre "
     "lo que queda. Al revés, el umbral se correría hacia abajo y se cortaría "
     "de más."),
    ("Columnas _bruto",
     "Las hojas de datos traen cada duración dos veces: la limpia, que es la "
     "que usan las fórmulas, y la bruta sin ningún filtro, para poder auditar "
     "qué se excluyó."),
    ("Estadísticos",
     "Son fórmulas sobre las hojas de datos de cada grupo, no valores "
     "pegados: al filtrar o corregir un dato, las tablas se recalculan."),
    ("", ""),
    ("ARTÍCULOS DE CADA GRUPO", ""),
]

for grupo, arts in GRUPOS.items():
    lineas.append((grupo, ", ".join(sorted(arts, key=lambda a: int(re.sub(r"\D", "", a))))))


lineas += [
    ("", ""),
    ("Nota sobre la partición",
     "El listado de referencia agrupa homicidios y delitos sexuales dentro de "
     "'Crimen violento'. La separación en dos grupos se hizo acá y se lista "
     "arriba artículo por artículo."),
    ("", ""),
    ("EXCLUSIONES POR VARIABLE (sobre la base total)", ""),
    ("variable", "válidas antes | negativas | sobre el umbral | umbral en días | válidas finales"),
]
for nombre, d in exclusiones.items():
    valores = list(d.values())
    lineas.append((
        nombre,
        f"{valores[0]:,} | {valores[1]:,} | {valores[2]:,} | "
        f"{valores[3]} | {valores[4]:,}",
    ))


for etiqueta, valor in lineas:
    f = doc.max_row + 1
    doc[f"A{f}"] = etiqueta
    doc[f"B{f}"] = valor
    doc[f"A{f}"].font = NEGRITA if etiqueta.isupper() and etiqueta else NORMAL
    doc[f"B{f}"].font = NORMAL
    doc[f"B{f}"].alignment = Alignment(wrap_text=True, vertical="top")


doc.column_dimensions["A"].width = 42
doc.column_dimensions["B"].width = 90

exc = wb.create_sheet("Exclusiones")
exc["A1"] = "Observaciones excluidas por tabla y variable"
exc["A1"].font = TITULO
exc["A2"] = (
    "Cuántas duraciones se cayeron de cada cálculo y por qué. El umbral es "
    "único por variable (calculado sobre la base total) y se aplica igual a "
    "las cuatro tablas."
)
exc["A2"].font = SUBTITULO

cabeceras_exc = ["Tabla", "Tiempo procesal", "Umbral (días)", "Válidas antes",
                 "Negativas", "Sobre el umbral", "Válidas finales", "% excluido"]
exc.append([])
exc.append(cabeceras_exc)

for celda in exc[4]:
    celda.font = NEGRITA
    celda.fill = PatternFill("solid", fgColor="EDEDED")
    celda.alignment = Alignment(horizontal="center", wrap_text=True)
    celda.border = BORDE_CABECERA


for grupo in TABLAS:
    sub = procesos_grupo[procesos_grupo["grupo"] == grupo]

    for etiqueta, variable, _, _ in INTERVALOS:
        bruto = sub[f"{variable}_bruto"]
        umbral = umbrales[variable]

        antes = int(bruto.notna().sum())
        negs = int((bruto < 0).sum())
        altos = int((bruto.notna() & (bruto >= 0) & (bruto > umbral)).sum()) if pd.notna(umbral) else 0
        final = int(sub[variable].notna().sum())

        exc.append([
            grupo, etiqueta,
            round(umbral, 1) if pd.notna(umbral) else None,
            antes, negs, altos, final,
            round((antes - final) / antes * 100, 1) if antes else None,
        ])


for fila in exc.iter_rows(min_row=5):
    for celda in fila:
        celda.font = NORMAL


exc.column_dimensions["A"].width = 22
exc.column_dimensions["B"].width = 40
for c in "CDEFGH":
    exc.column_dimensions[c].width = 15
exc.freeze_panes = "A5"
exc.auto_filter.ref = f"A4:H{exc.max_row}"

wb.save(RUTA_EXCEL)
print(f"Guardado: {RUTA_EXCEL}")
print("Hojas:", wb.sheetnames)
print("\nAl abrir el archivo, Excel calcula las fórmulas solo. Si lo leés con "
      "pandas antes de abrirlo, las celdas de fórmula salen vacías.")